In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score , GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd


os.chdir('/content/drive/MyDrive/TPSM-Project')


print(os.listdir())


df = pd.read_csv('dataset_cleaned.csv')


print("Loaded shape:", df.shape)
df.head()

['dataset_cleaned.csv', 'spotify_predictor_pipeline.pkl']
Loaded shape: (89740, 23)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,super_genre,popularity_tier
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,0,0.676,0.4610,...,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic,Acoustic,Hit
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,0,0.420,0.1660,...,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic,Acoustic,Average
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,0,0.438,0.3590,...,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic,Acoustic,Average
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,0,0.266,0.0596,...,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic,Acoustic,Hit
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,0,0.618,0.4430,...,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic,Acoustic,Hit


In [4]:
# Convert milliseconds to minutes
df["duration_min"] = df["duration_ms"] / 60000

In [5]:
print(df[["duration_ms", "duration_min"]].head())

   duration_ms  duration_min
0       230666      3.844433
1       149610      2.493500
2       210826      3.513767
3       201933      3.365550
4       198853      3.314217


In [6]:
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,super_genre,popularity_tier,duration_min
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,0,0.676,0.4610,...,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic,Acoustic,Hit,3.844433
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,0,0.420,0.1660,...,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic,Acoustic,Average,2.493500
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,0,0.438,0.3590,...,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic,Acoustic,Average,3.513767
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,0,0.266,0.0596,...,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic,Acoustic,Hit,3.365550
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,0,0.618,0.4430,...,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic,Acoustic,Hit,3.314217


During our feature selection phase, we deliberately dropped variables like 'key', 'mode', and 'time signature'. Features like time signature lack the mathematical variance needed for pattern recognition, as nearly all tracks default to 4/4 time. Furthermore, we wanted our predictive model to be strictly grounded in the sonic variables we mathematically validated during our Inferential T-Tests. This ensures our model runs on pure signal, rather than getting over-fitted by irrelevant sonic noise.

In [7]:
features = ["danceability", "energy", "loudness", "valence", "tempo", 
            "acousticness", "explicit", "duration_min", "super_genre"]
X = df[features]
y = df['popularity']

In [8]:
# 2. Train/Test Split (Done BEFORE any data manipulation to prevent leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# 3. Build the Preprocessing Engine
# We encode the genre (without drop_first) and let all numerical features pass through untouched (no scaling)
preprocessor = ColumnTransformer(
    transformers=[
        ('genre_encoder', OneHotEncoder(handle_unknown='ignore', drop=None), ['super_genre'])
    ],
    remainder='passthrough'
)

In [11]:

preprocessor.fit(X_train)


X_train_transformed = preprocessor.transform(X_train)

print("COLUMN TRANSFORMATION CHECK")
print("="*50)
print(f"Original shape: {X_train.shape} (Rows, Columns)")
print(f"New shape:      {X_train_transformed.shape} (Rows, Columns)")


new_columns = preprocessor.get_feature_names_out()

print("\nExact Column Names Fed to the Model:")
print("-" * 40)
for i, col in enumerate(new_columns):
    print(f"{i+1}. {col}")

COLUMN TRANSFORMATION CHECK
Original shape: (71792, 9) (Rows, Columns)
New shape:      (71792, 23) (Rows, Columns)

Exact Column Names Fed to the Model:
----------------------------------------
1. genre_encoder__super_genre_Acoustic
2. genre_encoder__super_genre_Alternative
3. genre_encoder__super_genre_Ambient
4. genre_encoder__super_genre_Classical
5. genre_encoder__super_genre_Country
6. genre_encoder__super_genre_Electronic
7. genre_encoder__super_genre_Hip-Hop
8. genre_encoder__super_genre_Jazz/Blues
9. genre_encoder__super_genre_Latin
10. genre_encoder__super_genre_Metal
11. genre_encoder__super_genre_Other
12. genre_encoder__super_genre_Pop
13. genre_encoder__super_genre_R&B/Soul
14. genre_encoder__super_genre_Rock
15. genre_encoder__super_genre_World
16. remainder__danceability
17. remainder__energy
18. remainder__loudness
19. remainder__valence
20. remainder__tempo
21. remainder__acousticness
22. remainder__explicit
23. remainder__duration_min


In [12]:
# 4. Build the Full Model Pipeline
# This bundles the encoder and the Random Forest into ONE seamless object
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])


In [13]:
param_grid = {
    'model__n_estimators': [100, 200],       # Try 100 trees, then try 200 trees
    'model__max_depth': [None, 15, 25],      # Try no depth limit, limit to 15 branches, limit to 25 branches
    'model__min_samples_split': [2, 5]       # Try requiring 2 samples to split, then try 5
}

In [14]:
print("Starting Grid Search...")

grid_search = GridSearchCV(
    pipeline, 
    param_grid=param_grid, 
    cv=3,                 # 3-Fold Cross Validation (to save a bit of time)
    scoring='r2',         # Tell the grid search to specifically maximize the R-squared score!
    n_jobs=-1,            # Use all available CPU cores on your computer
    verbose=2             # Print progress updates
)

Starting Grid Search...


In [15]:
grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('genre_encoder',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['super_genre'])])),
                                       ('model',
                                        RandomForestRegressor(n_jobs=-1,
                                                              random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [None, 15, 25],
                         'model__min_samples_split': [2, 5],
                         'model__n_estimators': [100, 200]},
             scoring='r2', verbose=2)

In [16]:
print("\n" + "="*50)
print("HYPERPARAMETER TUNING COMPLETE")
print("="*50)
print(f"Best Parameters Found:\n{grid_search.best_params_}")


HYPERPARAMETER TUNING COMPLETE
Best Parameters Found:
{'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 200}


In [17]:
best_pipeline = grid_search.best_estimator_
y_pred = best_pipeline.predict(X_test)

In [18]:
print("\nNEW TEST SET METRICS (With Tuned Model)")
print("-" * 40)
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R-squared (R2): {r2_score(y_test, y_pred):.4f}")


NEW TEST SET METRICS (With Tuned Model)
----------------------------------------
Mean Absolute Error (MAE): 12.67
R-squared (R2): 0.3133


In [19]:
joblib.dump(best_pipeline, 'spotify_predictor_pipeline_TUNED.pkl')
print("\n✓ SUCCESS: New Tuned Pipeline exported as 'spotify_predictor_pipeline_TUNED.pkl'")


✓ SUCCESS: New Tuned Pipeline exported as 'spotify_predictor_pipeline_TUNED.pkl'


In [13]:
# 5. Cross-Validation (Proving robustness for your presentation)
print("Running 5-Fold Cross Validation (This might take a moment)...")
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
print(f"CV Mean Absolute Error: {-cv_scores.mean():.2f} (+/- {cv_scores.std():.2f}) points")

Running 5-Fold Cross Validation (This might take a moment)...
CV Mean Absolute Error: 12.88 (+/- 0.08) points


In [14]:
# 6. Train the final pipeline on the training data
pipeline.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('genre_encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['super_genre'])])),
                ('model', RandomForestRegressor(n_jobs=-1, random_state=42))])

In [15]:
# 7. Evaluate on the untouched Test Set
y_pred = pipeline.predict(X_test)
print("\n" + "="*50)
print("TEST SET METRICS")
print("="*50)
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R-squared (R2): {r2_score(y_test, y_pred):.4f}")


TEST SET METRICS
Mean Absolute Error (MAE): 12.62
R-squared (R2): 0.3097


In [16]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Calculate MSE
mse = mean_squared_error(y_test, y_pred)

# Calculate RMSE (which is just the square root of MSE)
rmse = np.sqrt(mse)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

Mean Squared Error (MSE): 288.70
Root Mean Squared Error (RMSE): 16.99


In [17]:
# 9. Save the Pipeline for Deployment
joblib.dump(pipeline, 'spotify_predictor_pipeline.pkl')
print("\n✓ SUCCESS: Pipeline exported as 'spotify_predictor_pipeline.pkl'")


✓ SUCCESS: Pipeline exported as 'spotify_predictor_pipeline.pkl'


1. It Proves Stability (No Overfitting)
If your model scores an MAE of 5 on Round 1, but an MAE of 25 on Round 2, it means your model is unstable and memorizing data instead of learning patterns (this is called overfitting).

    Your Data: Because your Cross-Validation MAE (12.88) and your final Test MAE (12.62) were almost identical, it mathematically proves your model is highly stable and didn't overfit!